# W4Q2 — Marketing 12m Initiative — Before/After Analysis
**Question:** Did the marketing initiative targeting 12m subscriptions (`marketing_12m_subscr` flag) drive higher 12m conversion rates?

**Audience:** Marketing & Leadership  
**Data source:** `ANALYTICS.MARTS.MART_SUBSCRIPTION_FUNNEL`  
**SQL:** `sql/W4Q2_marketing_12m_initiative.sql`

---

> **Context:** `marketing_12m_subscr` is a flag indicating the user was targeted by a specific
> initiative aimed at driving annual subscriptions. This analysis compares 12m conversion
> outcomes for flagged vs non-flagged users.

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from src.connection import query

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130})
PALETTE = sns.color_palette('muted', 8)

## Load Data

In [ ]:
with open('../sql/W4Q2_marketing_12m_initiative.sql') as f:
    sql = f.read()

df = query(sql)
df['signup_date'] = pd.to_datetime(df['signup_date'])
print(f"Rows: {len(df):,}")
print(f"marketing_12m_subscr distribution:\n{df['marketing_12m_subscr'].value_counts(dropna=False)}")

## Conversion Outcomes — Flagged vs Not Flagged

In [ ]:
outcomes = {
    'has_trial': 'Trial Rate',
    'has_3m_subscription': '3m Rate',
    'has_12m_subscription': '12m Rate',
}

rows = []
for col, label in outcomes.items():
    for flag in df['marketing_12m_subscr'].dropna().unique():
        grp = df[df['marketing_12m_subscr'] == flag]
        n = len(grp)
        converted = grp[col].sum()
        rows.append({
            'marketing_12m_subscr': flag,
            'Outcome': label,
            'N': n,
            'Converted': converted,
            'Rate (%)': round(converted / n * 100, 1),
        })

comparison = pd.DataFrame(rows)
display(comparison)

In [ ]:
pivot = comparison.pivot(index='Outcome', columns='marketing_12m_subscr', values='Rate (%)')

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', ax=ax, color=PALETTE[:3], edgecolor='white')
ax.set_title('Conversion Rates by marketing_12m_subscr Flag', fontsize=13, pad=12)
ax.set_xlabel('')
ax.set_ylabel('Conversion Rate (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='marketing_12m_subscr')
sns.despine()
plt.tight_layout()
plt.savefig('../output/W4Q2_marketing_12m_conversion.png', bbox_inches='tight')
plt.show()

## 12m Conversion Over Time — Flagged vs Not Flagged

In [ ]:
df['signup_month'] = df['signup_date'].dt.to_period('M').dt.to_timestamp()

monthly = (
    df[df['signup_date'] >= '2020-01-01']
    .groupby(['signup_month', 'marketing_12m_subscr'])
    ['has_12m_subscription']
    .agg(['sum', 'count'])
    .assign(rate=lambda x: x['sum'] / x['count'] * 100)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 5))
for flag, grp in monthly.groupby('marketing_12m_subscr'):
    ax.plot(grp['signup_month'], grp['rate'], label=f'marketing_12m_subscr={flag}', linewidth=1.8)

ax.set_title('12m Conversion Rate Over Time — by marketing_12m_subscr Flag', fontsize=13, pad=12)
ax.set_xlabel('')
ax.set_ylabel('12m Conversion Rate (%)')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('../output/W4Q2_12m_rate_over_time.png', bbox_inches='tight')
plt.show()

## Findings

- **Flag distribution:** ...
- **12m conversion rate — flagged vs not flagged:** ...
- **3m and trial rates comparison:** ...
- **Trend over time:** ...
- **Statistical significance:** ...
- **Recommendation:** ...